In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb

# =========================
# 1) 載入資料
# =========================
data = pd.read_csv('feature_data/feature_data.csv')

# =========================
# 2) 資料預處理
# =========================
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    df_processed = df.copy()

    # 處理 spend_time 欄位（轉換為秒數）
    df_processed['spend_time_seconds'] = pd.to_timedelta(df_processed['spend_time']).dt.total_seconds()

    # 處理 high_elevation 布林值
    df_processed['high_elevation'] = df_processed['high_elevation'].fillna(False).astype(int)

    # 處理累計時間（轉換為秒數）
    df_processed['accumulated_time_seconds'] = pd.to_timedelta(df_processed['accumulated_time']).dt.total_seconds()

    # 處理 max_slope_point 座標字串
    df_processed['max_slope_lat'] = df_processed['max_slope_point'].str.extract(r'\(([^,]+),').astype(float)
    df_processed['max_slope_lon'] = df_processed['max_slope_point'].str.extract(r',\s*([^)]+)\)').astype(float)

    # 處理 slope_freq_dist 字串（轉換為數值特徵）
    slope_dist = df_processed['slope_freq_dist'].str.extract(r"'<-15°': ([^,]+), '-15°~-10°': ([^,]+), '-10°~-5°': ([^,]+), '-5°~-1°': ([^,]+), '-1°~1°': ([^,]+), '1°~5°': ([^,]+), '5°~10°': ([^,]+), '10°~15°': ([^,]+), '>15°': ([^}]+)")
    df_processed['slope_neg15'] = slope_dist[0].astype(float)
    df_processed['slope_neg15_neg10'] = slope_dist[1].astype(float)
    df_processed['slope_neg10_neg5'] = slope_dist[2].astype(float)
    df_processed['slope_neg5_neg1'] = slope_dist[3].astype(float)
    df_processed['slope_neg1_1'] = slope_dist[4].astype(float)
    df_processed['slope_1_5'] = slope_dist[5].astype(float)
    df_processed['slope_5_10'] = slope_dist[6].astype(float)
    df_processed['slope_10_15'] = slope_dist[7].astype(float)
    df_processed['slope_over15'] = slope_dist[8].astype(float)

    # 特徵集合（排除目標變數和不需要的欄位）
    features = ['avg_temp', 'avg_RH', 'max_precip', 'distance', 'elevation_range', 
                'elevation_change', 'elevation_gain', 'elevation_loss', 'high_elevation',
                'max_slope_percent', 'max_slope_degrees', 'slope_std_dev', 'slope_variance',
                'max_slope_lat', 'max_slope_lon', 'slope_neg15', 'slope_neg15_neg10', 
                'slope_neg10_neg5', 'slope_neg5_neg1', 'slope_neg1_1', 'slope_1_5', 
                'slope_5_10', 'slope_10_15', 'slope_over15', 'accumulated_time_seconds', 
                'accumulated_distance']

    return df_processed[features]

# 分割訓練和測試資料
from sklearn.model_selection import train_test_split

# 先處理資料以獲取 spend_time_seconds
data_processed = data.copy()
data_processed['spend_time_seconds'] = pd.to_timedelta(data_processed['spend_time']).dt.total_seconds()

X = preprocess_data(data)
y = data_processed['spend_time_seconds']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# =========================
# 3) 標準化
# =========================
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# =========================
# 4) XGBoost 參數與搜尋空間
# =========================
base_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'device': 'cuda:0',
    'verbosity': 0
}

param_dist = {
    'n_estimators': [100, 200, 300, 400, 500, 800, 1000],
    'max_depth': [3, 4, 5, 6, 7, 8, 10],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    'subsample': [0.8, 0.85, 0.9, 0.95, 1.0],
    'colsample_bytree': [0.8, 0.85, 0.9, 0.95, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.2, 0.3, 0.4],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [0, 0.1, 0.5, 1.0, 2.0]
}

# =========================
# 5) RandomizedSearchCV
# =========================
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

base_xgb = xgb.XGBRegressor(**base_params)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

random_search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train_scaled, y_train)

best_params = random_search.best_params_
best_score  = random_search.best_score_

print(f"最佳 RMSE: {(-best_score)**0.5:.4f}")

# =========================
# 6) 多模型集成
# =========================
models = []
for i in range(5):
    model = xgb.XGBRegressor(
        **base_params,
        **best_params,
        random_state=42 + i
    )
    model.fit(X_train_scaled, y_train)
    models.append(model)

def ensemble_predict(models_list, X):
    preds = [m.predict(X) for m in models_list]
    return np.mean(preds, axis=0)

train_pred = ensemble_predict(models, X_train_scaled)
train_rmse = mean_squared_error(y_train, train_pred, squared=False)
train_r2 = r2_score(y_train, train_pred)
print(f"訓練 RMSE: {train_rmse:.4f}, R²: {train_r2:.4f}")

# =========================
# 7) 測試集預測與評估
# =========================
test_pred = ensemble_predict(models, X_test_scaled)

test_rmse = mean_squared_error(y_test, test_pred, squared=False)
test_r2 = r2_score(y_test, test_pred)
print(f"測試 RMSE: {test_rmse:.4f}, R²: {test_r2:.4f}")

# 儲存預測結果
results = pd.DataFrame({
    'actual': y_test,
    'predicted': test_pred
})
results.to_csv('spend_time_predictions.csv', index=False)
print("完成")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
最佳 RMSE: 1186.8998
訓練 RMSE: 905.2510, R²: 0.9313
測試 RMSE: 1248.5907, R²: 0.8729
完成
